# 03 — System health & pipeline insights

**Audience:** engineering / on-call. Answers: is the data pipeline healthy, what's the end-to-end latency, are we dropping samples?

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from etl.config import load_config
from etl.extract import iter_documents
from etl.transform import to_dataframe, add_derived_features, session_summary
from etl.load import get_collection

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
config = load_config()

In [ ]:
collection = get_collection(config.mongo_uri, config.mongo_db, config.mongo_collection)
df = add_derived_features(to_dataframe(list(iter_documents(collection))))
print(f'{len(df):,} rows  |  {df.index.min()} → {df.index.max()}')

## 1. End-to-end latency (client → backend ingest)
`latency_ms = recorded_at - clientTimestamp`. High p95 indicates slow network or backend processing.

In [ ]:
lat = df['latency_ms'].dropna()
stats = pd.Series({
    'count': lat.count(),
    'mean': lat.mean(),
    'p50': lat.quantile(0.5),
    'p95': lat.quantile(0.95),
    'p99': lat.quantile(0.99),
    'max': lat.max(),
}).round(2)
print(stats)
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(lat.clip(lower=-200, upper=2000), bins=80, color='#0ea5e9', edgecolor='black', linewidth=0.3)
ax.axvline(lat.quantile(0.95), color='red', linestyle='--', label=f'p95={lat.quantile(0.95):.0f}ms')
ax.set_title('End-to-end latency distribution (clipped to -200..2000 ms)')
ax.set_xlabel('latency (ms)')
ax.legend()
plt.tight_layout()

## 2. Effective sample rate per session
After backend-side dedup (100 ms throttle + change-detection), the rate should be **below** 10 Hz.
If a session shows ~10 Hz, the dedup isn't kicking in for that session.

In [ ]:
summary = session_summary(df)
fig, ax = plt.subplots(figsize=(10, 4))
summary['effective_hz'].sort_values(ascending=False).head(30).plot.bar(ax=ax, color='#22c55e')
ax.axhline(10, color='red', linestyle='--', label='10 Hz cap')
ax.set_title('Effective sample rate per session (top 30 by rate)')
ax.set_ylabel('Hz')
ax.legend()
plt.xticks(rotation=70, ha='right')
plt.tight_layout()

## 3. Gap detection — when does the pipeline go silent?
Inter-sample gaps > 1 second inside a session usually mean network drop, backend hiccup, or driver paused.

In [ ]:
gaps = df['sample_dt_ms'].dropna()
long_gaps = gaps[gaps > 1000]
print(f'Total inter-sample gaps: {len(gaps):,}')
print(f'Gaps > 1s: {len(long_gaps):,}  ({len(long_gaps)/len(gaps)*100:.2f}%)')
if not long_gaps.empty:
    print(f'Worst: {long_gaps.max()/1000:.1f}s')
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(gaps.clip(upper=5000), bins=80, color='#f97316', edgecolor='black', linewidth=0.3)
ax.set_title('Inter-sample gap distribution (clipped at 5s)')
ax.set_xlabel('gap (ms)')
ax.set_yscale('log')
plt.tight_layout()

## 4. Source switchover events
Detect sessions where the driver switched between wheel and keyboard mid-run.

In [ ]:
switches = df[df['sessionId'].notna()].groupby('sessionId')['source'].nunique()
multi = switches[switches > 1]
print(f'Sessions with multiple sources: {len(multi)}')
multi.head(10)

## 5. Ingest volume over time
Useful for spotting outages or sudden surges.

In [ ]:
per_5min = df.resample('5min').size()
fig, ax = plt.subplots(figsize=(12, 4))
per_5min.plot(ax=ax, color='#6366f1', linewidth=0.8)
ax.set_title('Rows ingested per 5-minute window')
ax.set_ylabel('rows / 5min')
plt.tight_layout()

## Takeaways to write up
- p95 latency target (e.g. < 200 ms acceptable, > 500 ms warrants investigation)
- Whether the dedup achieved meaningful reduction (compare effective_hz against the raw 10 Hz input)
- Outage windows visible in section 5